### 15. 같은 구역에서 모델을 비교하고 점검할 구간 찾기

- 저장한 IsolationForest와 RandomForestClassifier 결과를 읽음
  - 모델을 다시 학습하거나 후보 기준을 바꾸지 않음
- 같은 CSV와 Train, Valid, Test의 LOT 분할을 사용하고 같은 구역과 평가 행에서 비교함
  - 보류를 `0`으로 채우지 않음
- 실제 학습에 넣는 행수는 모델마다 다름
  - IsolationForest는 구역과 작업 조건마다 최대 `MAX_FIT_ROWS`개를 뽑으며 기본값은 `12000`임
  - RandomForestClassifier는 입력값과 라벨이 있는 Train 전체를 사용함
- 라벨은 LOT 전체를 살펴본 뒤 만든 후보 규칙임
  - RandomForestClassifier 지표는 뒤 LOT에서 그 규칙을 얼마나 재현했는지 보여줌
  - IsolationForest는 라벨을 학습하지 않았지만 이 지표도 고장 정답과의 비교는 아님
- 기존 LOT별 Z-score는 같은 LOT 전체의 평균과 표준편차를 쓴 사후 기준임
  - 라벨 생성과 연결된 규칙이므로 독립적인 미래 예측 기준선으로 소개하지 않음


### 1. 파일과 볼 구역

- `DATA_FILE`만 바꾸면 해당 CSV로 완료한 두 모델의 실행을 찾음
- `PLOT_GROUP`은 첫 비교표에서 볼 구역임
- `069 Z4`와 `121 Z6`의 모델 후보는 뒤에서 따로 확인함
  - `VIEW_MARGIN_MINUTES`는 그림의 여백이며 탐지 기준이 아님


In [ ]:
DATA_FILE = "cal01_base_v3.csv"  # 1분 자료는 cal01_base_v4.csv
PLOT_GROUP = "Z3"  # 첫 결과표에서 볼 구역
VIEW_MARGIN_MINUTES = 10  # 선택 구간 앞뒤에 함께 그릴 시간

import os
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager, rcParams
from IPython.display import display, Markdown

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
ENC = "utf-8-sig"
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)
DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")
files = {
    "base": BASE_CSV,
    "event": os.path.join(DATA_DIR, "G-02_조업이벤트.csv"),
    "maintenance": os.path.join(DATA_DIR, "G-02_정비이력.csv"),
    "repair_plan": os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv"),
    "parts": os.path.join(DATA_DIR, "G-02_부품교체이력.csv"),
    "part_master": os.path.join(DATA_DIR, "G-01_부품마스터.csv"),
}
input_hashes = {}
for name, path in files.items():
    # 큰 CSV도 1MB씩 읽어 같은 내용 식별값을 계산함.
    source_hash = hashlib.sha256()
    with open(path, "rb") as source_file:
        chunk = source_file.read(1024 * 1024)
        while chunk:
            source_hash.update(chunk)
            chunk = source_file.read(1024 * 1024)
    input_hashes[name] = source_hash.hexdigest()
RUN_ID = os.path.splitext(DATA_FILE)[0] + "__" + input_hashes["base"][:8] + "__" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT_DIR = os.path.join(PROCESSED_DIR, "namelesspark", "model_review", RUN_ID)
FIG_DIR = os.path.join(ROOT, "figures", "namelesspark", "15_model_review", RUN_ID)
model_roots = {
    "IsolationForest": os.path.join(PROCESSED_DIR, "sayletskin", "isolation_forest_ab"),
    "RandomForestClassifier": os.path.join(PROCESSED_DIR, "sayletskin", "random_forest_classifier"),
}
config_names = {"IsolationForest": "settings.json", "RandomForestClassifier": "run_config.json"}
groups = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}
method_names = {
    "fixed_9_2_3": "고정 9/2/3",
    "same_grade_history": "이전 동일 강종",
    "all_history": "이전 모든 강종",
    "first_lot_60_20_20": "001 시간 60/20/20",
}
SENSORS = []
for sensors in groups.values():
    SENSORS.extend(sensors)
font_names = []
for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in font_names:
        rcParams["font.family"] = font_name
        break
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

header = pd.read_csv(BASE_CSV, encoding=ENC, nrows=0).columns.tolist()
DATA_KIND = "minute" if "BIN_START" in header else "raw"
base_columns = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED", "EXCLUSION_REASON"] + SENSORS
for group in groups:
    base_columns.append(group + "__LABEL")
    if group + "__COMMON_REVIEW_IDS" in header:
        base_columns.append(group + "__COMMON_REVIEW_IDS")
for column in ["OBS_START", "OBS_END", "AVAILABLE_AT", "SEGMENT", "BIN_START", "BIN_END", "ROW_COUNT"]:
    if column in header:
        base_columns.append(column)
base = pd.read_csv(BASE_CSV, encoding=ENC, usecols=base_columns, dtype={"ROW_ID": "string", "LOT_NO": "string"}, low_memory=False)
base["MEAS_DT"] = pd.to_datetime(base["MEAS_DT"])
base["EXCLUDED"] = ~base["EXCLUDED"].astype("string").str.strip().str.lower().isin(["false", "0", "0.0"])
base["EXCLUSION_REASON"] = base["EXCLUSION_REASON"].fillna("")
if DATA_KIND == "raw":
    base["OBS_START"] = base["MEAS_DT"]
    base["OBS_END"] = base["MEAS_DT"]
    base["AVAILABLE_AT"] = base["MEAS_DT"]
else:
    for column in ["OBS_START", "OBS_END", "AVAILABLE_AT", "BIN_START", "BIN_END"]:
        base[column] = pd.to_datetime(base[column])
for group in groups:
    base[group + "__LABEL"] = pd.to_numeric(base[group + "__LABEL"], errors="coerce")
base = base.sort_values(["LOT_NO", "OBS_START"], kind="stable").reset_index(drop=True)
base_by_row = base.set_index("ROW_ID")
print("입력:", BASE_CSV, "| 행 단위:", "1분 안의 관측 구간" if DATA_KIND == "minute" else "원본 기록")


### 2. 같은 자료로 완료한 실행 선택

- CSV와 조업 원본 3개의 내용이 같은 완료 실행만 선택함
  - 구역, 입력 컬럼, 학습 LOT 목록도 맞춰 봄
  - 과거창·기록 공백·기준 최소 기록 수와 실제 분할 시각을 맞춰 봄
  - v4는 같은 CSV에 계산된 과거 특징을 사용하므로 기준 최소 기록 수와 관측 분을 비교함
- 학습 표본 수·나무 수·후보 기준은 모델마다 다를 수 있음
  - 선택한 실행의 나무 수와 후보 기준을 아래에 표시함
- 사용한 실행 폴더를 아래에 표시함
  - 맞는 실행이 없으면 비교를 진행하지 않음
- v4를 만들 때 사용한 조업 원본도 확인함
  - 원본이 나중에 바뀌었다면 현재 정비 기록과의 대조를 별도로 표시함


In [ ]:
# 같은 뜻으로 저장된 옵션 이름을 맞춤. 학습 표본 수와 모델별 후보 기준은 비교하지 않음.
shared_options = {
    "기준 최소 기록 수": ("reference_min_rows", "minimum_rows"),
    "기준 최소 관측 분": ("reference_min_observed_minutes", "minimum_observed_minutes"),
}
if DATA_KIND == "raw":
    shared_options.update({
        "과거창(초)": ("rolling_seconds", "raw_window_seconds"),
        "과거창 포함 범위": ("rolling_closed", "raw_window_closed"),
        "과거창 최소 기록 수": ("rolling_min_past_rows", "raw_min_past_rows"),
        "기록 공백(초)": ("gap_reset_seconds", "raw_gap_seconds"),
    })

run_catalog = {"IsolationForest": [], "RandomForestClassifier": []}
run_checks = []
for model_name, model_root in model_roots.items():
    if not os.path.isdir(model_root):
        continue
    for folder_name in os.listdir(model_root):
        folder = os.path.join(model_root, folder_name)
        if not os.path.isdir(folder):
            continue
        check = {"MODEL": model_name, "RUN": folder_name, "SOURCES_MATCH": None, "COMPATIBLE": False, "REASON": ""}
        run_checks.append(check)
        config_path = os.path.join(folder, config_names[model_name])
        prediction_path = os.path.join(folder, "predictions.csv")
        plan_path = os.path.join(folder, "split_plan.csv")
        if not os.path.isfile(config_path) or not os.path.isfile(prediction_path) or not os.path.isfile(plan_path):
            check["REASON"] = "설정·예측·분할 파일 중 아직 없는 파일이 있음"
            continue
        if model_name == "RandomForestClassifier" and not os.path.isfile(os.path.join(folder, "metrics.csv")):
            check["REASON"] = "반복 지표 파일이 아직 없음"
            continue
        try:
            with open(config_path, encoding="utf-8") as config_file:
                config = json.load(config_file)
        except (json.JSONDecodeError, UnicodeDecodeError):
            check["REASON"] = "설정 JSON의 저장이 끝나지 않았거나 내용이 깨짐"
            continue
        hashes = config.get("input_sha256", {})
        if isinstance(hashes, str):
            hashes = {"base": hashes}
        same_sources = True
        for name in ["base", "event", "maintenance", "repair_plan"]:
            if hashes.get(name) != input_hashes[name]:
                same_sources = False
        check["SOURCES_MATCH"] = same_sources
        if not same_sources or config.get("data_kind") != DATA_KIND or config.get("groups") != groups:
            check["REASON"] = "CSV·조업 원본·행 단위·구역 구성 중 다른 항목이 있음"
            continue
        preprocessing = {}
        for name, (if_key, rfc_key) in shared_options.items():
            preprocessing[name] = config.get(if_key) if model_name == "IsolationForest" else config.get("reference", {}).get(rfc_key)
        if any(value is None for value in preprocessing.values()):
            check["REASON"] = "비교에 필요한 전처리 옵션이 설정 파일에 없음"
            continue
        plan = pd.read_csv(plan_path, encoding=ENC, dtype={"LOT_NO": "string"})
        required_plan = ["CASE", "ROLE", "LOT_NO", "TRAIN_LOTS", "START", "END", "TOTAL_ROWS"]
        if any(column not in plan for column in required_plan):
            check["REASON"] = "분할 파일에 LOT·시각·전체 행수 컬럼이 부족함"
            continue
        # IF의 AVAILABLE_END와 RFC의 END는 해당 구간의 값을 모두 사용할 수 있는 끝 시각임.
        end_column = "AVAILABLE_END" if "AVAILABLE_END" in plan else "END"
        plan["COMPARE_START"] = pd.to_datetime(plan["START"]).astype("string").fillna("")
        plan["COMPARE_END"] = pd.to_datetime(plan[end_column]).astype("string").fillna("")
        case_plans = {}
        for case, rows in plan.groupby("CASE", sort=False):
            boundaries = []
            for row in rows.itertuples():
                boundaries.append((str(row.ROLE), str(row.LOT_NO), row.COMPARE_START, row.COMPARE_END, int(row.TOTAL_ROWS)))
            # TOTAL_ROWS는 분할 전체 기록 수임. 모델별 유효 행수나 학습 표본 수가 아님.
            case_plans[case] = {"train_lots": sorted(rows["TRAIN_LOTS"].fillna("").unique().tolist()), "boundaries": sorted(boundaries)}
        with open(config_path, "rb") as config_file:
            config_hash = hashlib.sha256(config_file.read()).hexdigest()
        check["COMPATIBLE"] = True
        check["REASON"] = "자료와 필수 설정을 확인함"
        run_catalog[model_name].append({
            "folder": folder, "run_id": config.get("run_id", folder_name), "config": config,
            "config_sha256": config_hash, "case_plans": case_plans, "preprocessing": preprocessing,
            "features": config.get("group_input_columns", config.get("input_columns_by_group", {})),
            "started": config.get("run_id", folder_name).rsplit("__", 1)[-1],
        })
    run_catalog[model_name] = sorted(run_catalog[model_name], key=lambda run: run["started"], reverse=True)

selected_runs = {}
common_cases = []
pair_checks = []
for rfc_run in run_catalog["RandomForestClassifier"]:
    for if_run in run_catalog["IsolationForest"]:
        reasons = []
        if rfc_run["features"] != if_run["features"]:
            reasons.append("입력 컬럼이 다름")
        for name in shared_options:
            if rfc_run["preprocessing"][name] != if_run["preprocessing"][name]:
                reasons.append(name + " 설정이 다름")
        shared_cases = sorted(set(rfc_run["case_plans"]).intersection(if_run["case_plans"]))
        if not shared_cases:
            reasons.append("공통 실험이 없음")
        for case in shared_cases:
            if rfc_run["case_plans"][case] != if_run["case_plans"][case]:
                reasons.append(case + ": Train·Valid·Test의 LOT·시각·전체 행수가 다름")
        pair_checks.append({"IsolationForest": if_run["run_id"], "RandomForestClassifier": rfc_run["run_id"], "비교 가능": not reasons, "이유": "; ".join(reasons) or "자료·전처리·시간 분할이 같음"})
        if not reasons:
            selected_runs = {"IsolationForest": if_run, "RandomForestClassifier": rfc_run}
            common_cases = shared_cases
            break
    if selected_runs:
        break
RUN_READY = bool(selected_runs)
context_generation = "원본 기록 시각에 현재 조업 원본을 연결한 실행"
if DATA_KIND == "minute":
    context_generation = "v4 생성 원본 확인 필요"
    metadata_path = os.path.join(PROCESSED_DIR, "metadata", "v4", "dataset.json")
    if os.path.isfile(metadata_path):
        with open(metadata_path, encoding="utf-8") as metadata_file:
            minute_metadata = json.load(metadata_file)
        original_hashes = minute_metadata.get("source_sha256", {})
        context_matches = minute_metadata.get("master_sha256") == input_hashes["base"]
        for name in ["event", "maintenance", "repair_plan"]:
            if original_hashes.get(name) != input_hashes[name]:
                context_matches = False
        context_generation = "v4 생성에 사용한 조업 원본과 현재 원본이 같음" if context_matches else "현재 조업 원본이 v4 생성 원본과 다름. 아래 기록 대조는 갱신된 원본을 사용함"

selected_table = []
selected_settings = []
for model_name, run in selected_runs.items():
    config = run["config"]
    selected_table.append({"모델": model_name, "사용한 실행 폴더": run["folder"], "행 단위": DATA_KIND})
    if model_name == "IsolationForest":
        quantile = config.get("train_quantile")
        rule = f"Train 점수의 {100 * quantile:g}% 지점 초과" if quantile is not None else "설정 기록 없음"
        trees = config.get("n_estimators")
    else:
        rule = "예측 확률 ≥ " + str(config.get("parameters", {}).get("probability_threshold"))
        trees = config.get("parameters", {}).get("n_estimators")
    selected_settings.append({"모델": model_name, "나무 수": trees, "후보 기준": rule})
if RUN_READY:
    os.makedirs(OUT_DIR)
    os.makedirs(FIG_DIR)
    display(pd.DataFrame(selected_table))
    display(pd.DataFrame(selected_settings))
    display(pd.DataFrame(selected_runs["IsolationForest"]["preprocessing"].items(), columns=["공통 전처리 옵션", "설정값"]))
    print(context_generation)
    pd.DataFrame(run_checks).to_csv(os.path.join(OUT_DIR, "run_selection.csv"), index=False, encoding=ENC)
    pd.DataFrame(pair_checks).to_csv(os.path.join(OUT_DIR, "run_pair_selection.csv"), index=False, encoding=ENC)
else:
    print("같은 자료·전처리·시간 분할로 완료한 두 모델 실행이 없음. 아래 이유를 확인한 뒤 필요한 모델만 다시 실행할 것.")
    display(pd.DataFrame(run_checks).rename(columns={"MODEL": "모델", "RUN": "실행", "SOURCES_MATCH": "원본 일치", "COMPATIBLE": "필수 설정 확인", "REASON": "이유"}))
    if pair_checks:
        display(pd.DataFrame(pair_checks))


### 3. 같은 행에서 네 가지 예측 비교

- 구역 라벨과 네 예측이 모두 있는 행만 공통 비교에 사용함
  - 비교에서 빠진 전체 행과 라벨 `1`도 따로 셈
- 두 Dummy는 RandomForestClassifier 노트북이 저장한 예측을 그대로 읽음.
  - 이 표의 무작위 Dummy는 시드 `0` 한 번의 결과임.
  - 무작위 예측은 Train의 구역 라벨 비율을 사용하며 반복 평균은 다음 표에서 따로 봄.
- `Precision`은 후보로 표시한 행 중 기존 라벨 `1`의 비율임
- `Recall`은 공통 비교에 남은 라벨 `1` 중 후보로 찾은 비율임
  - 예측 `1`이 없으면 Precision, 기존 `1`이 없으면 Recall을 비워 둠


In [ ]:
if RUN_READY:
    prediction_tables = {}
    for model_name, run in selected_runs.items():
        prediction_path = os.path.join(run["folder"], "predictions.csv")
        prediction_header = pd.read_csv(prediction_path, encoding=ENC, nrows=0).columns.tolist()
        columns = ["CASE", "METHOD", "ROLE", "ROW_ID", "LOT_NO", "MEAS_DT", "CONTEXT"]
        if "SEGMENT" in prediction_header:
            columns.append("SEGMENT")
        for group in groups:
            for suffix in ["LABEL", "FLAG", "STATUS", "DUMMY0", "DUMMY_RANDOM0"]:
                column = group + "__" + suffix
                if column in prediction_header:
                    columns.append(column)
        predictions = pd.read_csv(
            prediction_path, encoding=ENC, usecols=columns,
            dtype={"ROW_ID": "string", "LOT_NO": "string", "CASE": "category", "METHOD": "category", "ROLE": "category"},
            low_memory=False,
        )
        # 한쪽에만 있는 추가 실험은 빼되, 모든 행이 공통이면 표를 다시 복사하지 않음.
        common_case = predictions["CASE"].isin(common_cases)
        if not common_case.all():
            predictions = predictions.loc[common_case].copy()
        predictions["MEAS_DT"] = pd.to_datetime(predictions["MEAS_DT"])
        if predictions[["CASE", "ROLE", "ROW_ID"]].duplicated().any():
            RUN_READY = False
            print(model_name, "결과에 같은 평가 행이 중복됨. 비교를 보류함.")
        prediction_tables[model_name] = predictions

metric_rows = []
coverage_rows = []
common_row_hashes = []
if RUN_READY:
    if_predictions = prediction_tables["IsolationForest"]
    rfc_predictions = prediction_tables["RandomForestClassifier"]
    keys = ["CASE", "ROLE", "ROW_ID"]
    for group in groups:
        left = if_predictions[keys + ["METHOD", "LOT_NO", "MEAS_DT", "CONTEXT", group + "__LABEL", group + "__FLAG", group + "__STATUS"]].copy()
        left = left.rename(columns={"LOT_NO": "IF_LOT", "MEAS_DT": "IF_TIME", "CONTEXT": "IF_CONTEXT", group + "__LABEL": "IF_LABEL", group + "__FLAG": "IF", group + "__STATUS": "IF_STATUS"})
        right = rfc_predictions[keys + ["LOT_NO", "MEAS_DT", "CONTEXT", group + "__LABEL", group + "__FLAG", group + "__STATUS", group + "__DUMMY0", group + "__DUMMY_RANDOM0"]].copy()
        right = right.rename(columns={"LOT_NO": "RFC_LOT", "MEAS_DT": "RFC_TIME", "CONTEXT": "RFC_CONTEXT", group + "__LABEL": "RFC_LABEL", group + "__FLAG": "RFC", group + "__STATUS": "RFC_STATUS", group + "__DUMMY0": "DUMMY0", group + "__DUMMY_RANDOM0": "DUMMY_RANDOM0"})
        joined = left.merge(right, on=keys, how="outer")
        joined["LOT_NO"] = joined["ROW_ID"].map(base_by_row["LOT_NO"])
        joined["LABEL"] = joined["ROW_ID"].map(base_by_row[group + "__LABEL"])
        joined["BASE_TIME"] = joined["ROW_ID"].map(base_by_row["MEAS_DT"])
        label_match = joined["LABEL"].eq(joined["IF_LABEL"]) & joined["LABEL"].eq(joined["RFC_LABEL"])
        metadata_match = joined["IF_LOT"].eq(joined["LOT_NO"]) & joined["RFC_LOT"].eq(joined["LOT_NO"])
        metadata_match = metadata_match & joined["IF_TIME"].eq(joined["BASE_TIME"]) & joined["RFC_TIME"].eq(joined["BASE_TIME"])
        metadata_match = metadata_match & joined["IF_CONTEXT"].eq(joined["RFC_CONTEXT"])
        joined["COMMON"] = joined["LABEL"].isin([0, 1]) & label_match & metadata_match
        for column in ["IF", "RFC", "DUMMY0", "DUMMY_RANDOM0"]:
            joined["COMMON"] = joined["COMMON"] & joined[column].isin([0, 1])
        joined["MATCHED_METADATA"] = metadata_match & label_match

        for (case, role), case_rows in joined.groupby(["CASE", "ROLE"], observed=True, sort=False):
            scopes = [("전체", case_rows)]
            for lot, lot_rows in case_rows.groupby("LOT_NO", sort=False):
                scopes.append((lot, lot_rows))
            for lot, rows in scopes:
                common = rows.loc[rows["COMMON"]]
                method_values = rows["METHOD"].dropna().astype(str).unique()
                method = method_values[0] if len(method_values) else ""
                reasons = []
                for model_name, flag_col, status_col in [("IsolationForest", "IF", "IF_STATUS"), ("RandomForestClassifier", "RFC", "RFC_STATUS")]:
                    for status, count in rows.loc[rows[flag_col].isna(), status_col].fillna("연결된 결과 없음").value_counts().items():
                        reasons.append(model_name + ": " + str(status) + " " + str(count) + "행")
                unknown = int((~rows["LABEL"].isin([0, 1])).sum())
                if unknown:
                    reasons.append("구역 라벨 보류 " + str(unknown) + "행")
                mismatched = int((rows["LABEL"].isin([0, 1]) & ~rows["MATCHED_METADATA"]).sum())
                if mismatched:
                    reasons.append("시각, 작업 조건 또는 라벨 연결 불일치 " + str(mismatched) + "행")
                row_hash = hashlib.sha256("\n".join(sorted(common["ROW_ID"].astype(str).tolist())).encode()).hexdigest()
                common_row_hashes.append({"CASE": case, "ROLE": role, "LOT_NO": lot, "GROUP": group, "ROWS": len(common), "SHA256": row_hash})
                coverage_rows.append({
                    "CASE": case, "METHOD": method, "ROLE": role, "LOT_NO": lot, "GROUP": group,
                    "ALL_ROWS": len(rows), "LABEL_ROWS": int(rows["LABEL"].isin([0, 1]).sum()),
                    "IF_SCORED": int(rows["IF"].notna().sum()), "RFC_SCORED": int(rows["RFC"].notna().sum()),
                    "COMMON_ROWS": len(common), "COVERAGE": len(common) / len(rows) if len(rows) else np.nan,
                    "NOT_COMPARED": len(rows) - len(common), "LABEL1_ALL": int(rows["LABEL"].eq(1).sum()),
                    "LABEL1_COMPARED": int(common["LABEL"].eq(1).sum()),
                    "LABEL1_NOT_COMPARED": int(rows["LABEL"].eq(1).sum() - common["LABEL"].eq(1).sum()),
                    "REASON": ", ".join(reasons),
                })
                for model_name, flag_column in [("IsolationForest", "IF"), ("RandomForestClassifier", "RFC"), ("Dummy 0 고정", "DUMMY0"), ("Dummy 무작위 시드 0", "DUMMY_RANDOM0")]:
                    y, pred = common["LABEL"], common[flag_column]
                    tp = int((y.eq(1) & pred.eq(1)).sum())
                    fp = int((y.eq(0) & pred.eq(1)).sum())
                    fn = int((y.eq(1) & pred.eq(0)).sum())
                    tn = int((y.eq(0) & pred.eq(0)).sum())
                    denominator = 2 * tp + fp + fn
                    f1 = 2 * tp / denominator if denominator else 0.0
                    metric_rows.append({
                        "CASE": case, "METHOD": method, "ROLE": role, "LOT_NO": lot, "GROUP": group,
                        "MODEL": model_name, "ROWS": len(common), "LABEL1": int(y.eq(1).sum()),
                        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                        "PRECISION": tp / (tp + fp) if tp + fp else np.nan,
                        "RECALL": tp / (tp + fn) if tp + fn else np.nan,
                        "F1": f1 if len(common) else np.nan, "CANDIDATE_RATE": pred.mean(),
                    })
    metrics = pd.DataFrame(metric_rows)
    coverage = pd.DataFrame(coverage_rows)
    metrics.to_csv(os.path.join(OUT_DIR, "comparison.csv"), index=False, encoding=ENC)
    coverage.to_csv(os.path.join(OUT_DIR, "coverage.csv"), index=False, encoding=ENC)
    show_names = {
        "METHOD": "학습 방법", "ROLE": "구간", "LOT_NO": "LOT", "GROUP": "구역", "MODEL": "모델",
        "ROWS": "공통 비교 행", "LABEL1": "기존 구역 1", "TP": "찾은 구역 1", "FP": "기존 구역 0 중 후보", "FN": "놓친 구역 1",
        "ALL_ROWS": "전체 평가 행", "LABEL_ROWS": "라벨이 있는 행", "IF_SCORED": "IsolationForest 판정 행", "RFC_SCORED": "RandomForestClassifier 판정 행",
        "COMMON_ROWS": "공통 비교 행", "COVERAGE": "비교한 비율", "NOT_COMPARED": "비교에서 빠진 행",
        "LABEL1_ALL": "전체 구역 1", "LABEL1_COMPARED": "비교한 구역 1", "LABEL1_NOT_COMPARED": "비교에서 빠진 구역 1", "REASON": "비교에서 빠진 이유",
    }
    metric_columns = ["METHOD", "ROLE", "LOT_NO", "MODEL", "ROWS", "LABEL1", "TP", "FP", "FN", "PRECISION", "RECALL", "F1"]
    display(metrics.loc[metrics["GROUP"].eq(PLOT_GROUP) & metrics["CASE"].eq("fixed_9_2_3") & metrics["LOT_NO"].ne("전체"), metric_columns].replace({"METHOD": method_names}).rename(columns=show_names).round(4))
    display(coverage.loc[coverage["GROUP"].eq(PLOT_GROUP) & coverage["CASE"].eq("fixed_9_2_3") & coverage["LOT_NO"].ne("전체")].drop(columns="CASE").replace({"METHOD": method_names}).rename(columns=show_names))


### 4. 무작위 Dummy의 반복 결과는 따로 보기

- 아래 표는 RandomForestClassifier 노트북이 이미 저장한 반복 결과임
  - `RUNS`는 실제 반복 횟수임
  - 위 공통 비교표보다 평가 행이 많을 수 있음
  - `RandomForestClassifier 비교 행`과 `IsolationForest까지 맞춘 공통 행`을 함께 표시함
- 반복의 퍼짐은 같은 자료에서 난수만 바꾼 차이임
  - 미래 성능의 신뢰구간을 뜻하지 않음


In [ ]:
if RUN_READY:
    rfc_metrics = pd.read_csv(os.path.join(selected_runs["RandomForestClassifier"]["folder"], "metrics.csv"), encoding=ENC, dtype={"LOT_NO": "string"})
    dummy_summary = rfc_metrics.loc[rfc_metrics["MODEL"].eq("Dummy 무작위")].copy()
    summary_keys = ["CASE", "ROLE", "LOT_NO", "GROUP"]
    dummy_summary = dummy_summary.merge(coverage[summary_keys + ["COMMON_ROWS"]], on=summary_keys, how="left")
    dummy_summary["DENOMINATOR_NOTE"] = "RandomForestClassifier가 판정한 행의 반복 결과이며 위 공통 비교와 행 수가 다를 수 있음"
    dummy_summary.to_csv(os.path.join(OUT_DIR, "dummy_repeats_reference.csv"), index=False, encoding=ENC)
    summary_columns = ["METHOD", "ROLE", "LOT_NO", "N_ROWS", "COMMON_ROWS", "RUNS", "PRECISION_MEAN", "PRECISION_STD", "RECALL_MEAN", "RECALL_STD", "F1_MEAN", "F1_STD", "F1_MIN", "F1_MAX"]
    display(dummy_summary.loc[dummy_summary["GROUP"].eq(PLOT_GROUP) & dummy_summary["CASE"].eq("fixed_9_2_3") & dummy_summary["LOT_NO"].ne("전체"), summary_columns].replace({"METHOD": method_names}).rename(columns={"N_ROWS": "RandomForestClassifier 비교 행", "COMMON_ROWS": "IsolationForest까지 맞춘 공통 행", "RUNS": "반복 횟수", "LOT_NO": "LOT"}).round(4))


### 5. 같은 LOT 전체로 계산한 Z-score를 나란히 보기

- 원본 기록에서 같은 LOT의 센서별 `평균 ±3σ`를 계산함
  - 기존 제외 행을 빼고 `ddof=0`, `|Z| > 3`을 사용함
  - `ddof=0`은 표준편차를 구할 때 유효한 기록 수로 나누는 옵션임
  - 평균에서 위아래 표준편차의 `3배` 범위를 벗어나면 후보
  - 공통 패턴을 후보에서 빼기 전의 결과임
- 구역의 센서 점수가 모두 있으면 그 구역 안의 초과 여부를 표시함
  - 한 센서라도 계산할 수 없으면 보류
- v4는 같은 원본 기록의 후보를 연결표로 모음
  - 분 안에 원본 기록 후보가 하나라도 있으면 `1`
  - 모두 `0`일 때만 `0`, 나머지는 보류
  - 분 평균으로 Z-score를 다시 계산하지 않음
- 아래 표는 라벨을 만든 이유를 확인하는 사후 비교임
  - 해당 LOT 전체를 사용하므로 미래 예측 순위에 섞지 않음
  - 모델의 공통 평가 행수와 분모가 다르며 점검 그림에는 같은 시각으로 붙임


In [ ]:
raw_trace = pd.DataFrame()
retro_flags = pd.DataFrame(np.nan, index=base["ROW_ID"], columns=list(groups))
sigma_reference = pd.DataFrame()
retro_summary = pd.DataFrame()
retro_status = "원본 기록과 생성 정보를 확인하지 못해 Z-score 사후 비교 보류"
if RUN_READY:
    raw_path = BASE_CSV
    expected_raw_hash = input_hashes["base"]
    raw_ready = DATA_KIND == "raw"
    if DATA_KIND == "minute":
        raw_ready = os.path.isfile(metadata_path) and minute_metadata.get("master_sha256") == input_hashes["base"]
        if raw_ready:
            raw_path = os.path.join(PROCESSED_DIR, minute_metadata.get("source_file", ""))
            expected_raw_hash = minute_metadata.get("source_sha256", {}).get("base")
    raw_metadata_path = os.path.join(PROCESSED_DIR, "metadata", "v3", "dataset.json")
    raw_ready = raw_ready and os.path.isfile(raw_path) and os.path.isfile(raw_metadata_path)
    if raw_ready:
        raw_hasher = hashlib.sha256()
        with open(raw_path, "rb") as raw_file:
            chunk = raw_file.read(1024 * 1024)
            while chunk:
                raw_hasher.update(chunk)
                chunk = raw_file.read(1024 * 1024)
        raw_hash = raw_hasher.hexdigest()
        with open(raw_metadata_path, encoding="utf-8") as raw_metadata_file:
            raw_metadata = json.load(raw_metadata_file)
        raw_ready = raw_hash == expected_raw_hash == raw_metadata.get("master_sha256")
        raw_ready = raw_ready and raw_metadata.get("groups") == groups
    if raw_ready:
        input_hashes["zscore_source_raw"] = raw_hash
        if DATA_KIND == "raw":
            raw_trace = base
        else:
            raw_columns = ["ROW_ID", "LOT_NO", "MEAS_DT", "EXCLUDED"] + SENSORS
            for group in groups:
                raw_columns.extend([group + "__LABEL", group + "__COMMON_REVIEW_IDS"])
            raw_trace = pd.read_csv(raw_path, encoding=ENC, usecols=raw_columns, dtype={"ROW_ID": "string", "LOT_NO": "string"}, low_memory=False)
            raw_trace["MEAS_DT"] = pd.to_datetime(raw_trace["MEAS_DT"])
            raw_trace["EXCLUDED"] = ~raw_trace["EXCLUDED"].astype("string").str.strip().str.lower().isin(["false", "0", "0.0"])

        # 연결표의 각 원행이 실제 같은 LOT와 시각인지 확인함
        if DATA_KIND == "minute":
            map_path = os.path.join(PROCESSED_DIR, minute_metadata.get("row_map", ""))
            raw_ready = os.path.isfile(map_path)
            if raw_ready:
                row_map = pd.read_csv(map_path, encoding=ENC, dtype={"SOURCE_ROW_ID": "string", "MINUTE_ROW_ID": "string", "LOT_NO": "string"})
                row_map["SOURCE_MEAS_DT"] = pd.to_datetime(row_map["SOURCE_MEAS_DT"])
                row_map["BIN_START"] = pd.to_datetime(row_map["BIN_START"])
                raw_index = raw_trace.set_index("ROW_ID")
                raw_ready = len(row_map) == len(raw_trace) and row_map["SOURCE_ROW_ID"].is_unique
                raw_ready = raw_ready and row_map["SOURCE_ROW_ID"].isin(raw_index.index).all()
                raw_ready = raw_ready and row_map["MINUTE_ROW_ID"].isin(base_by_row.index).all()
                raw_ready = raw_ready and row_map["LOT_NO"].eq(row_map["SOURCE_ROW_ID"].map(raw_index["LOT_NO"])).all()
                raw_ready = raw_ready and row_map["SOURCE_MEAS_DT"].eq(row_map["SOURCE_ROW_ID"].map(raw_index["MEAS_DT"])).all()
                for column in ["LOT_NO", "SEGMENT", "BIN_START"]:
                    raw_ready = raw_ready and row_map[column].eq(row_map["MINUTE_ROW_ID"].map(base_by_row[column])).all()
                mapped_ranges = row_map.groupby("MINUTE_ROW_ID").agg(OBS_START=("SOURCE_MEAS_DT", "min"), OBS_END=("SOURCE_MEAS_DT", "max"), ROW_COUNT=("SOURCE_ROW_ID", "size"))
                raw_ready = raw_ready and len(mapped_ranges) == len(base)
                for column in ["OBS_START", "OBS_END", "ROW_COUNT"]:
                    raw_ready = raw_ready and mapped_ranges[column].eq(base_by_row[column].reindex(mapped_ranges.index)).all()
                if raw_ready:
                    with open(map_path, "rb") as map_file:
                        input_hashes["raw_to_minute"] = hashlib.sha256(map_file.read()).hexdigest()
        if raw_ready:
            values = raw_trace[SENSORS].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
            not_excluded = ~raw_trace["EXCLUDED"]
            reference = values.loc[not_excluded]
            lot_keys = raw_trace.loc[not_excluded, "LOT_NO"]
            lot_mean = reference.groupby(lot_keys).mean()
            lot_std = reference.groupby(lot_keys).std(ddof=0)
            lot_count = reference.groupby(lot_keys).count()
            scores = pd.DataFrame(np.nan, index=raw_trace.index, columns=SENSORS)
            reference_rows = []
            for sensor in SENSORS:
                mean = raw_trace["LOT_NO"].map(lot_mean[sensor])
                std = raw_trace["LOT_NO"].map(lot_std[sensor]).replace(0, np.nan)
                scores[sensor] = values[sensor].subtract(mean).divide(std).abs().where(not_excluded)
                for lot in lot_mean.index:
                    reference_rows.append({"LOT_NO": lot, "SENSOR": sensor, "ROWS": int(lot_count.loc[lot, sensor]), "MEAN": lot_mean.loc[lot, sensor], "STD_DDOF0": lot_std.loc[lot, sensor]})
            for group, sensors in groups.items():
                available = np.isfinite(scores[sensors]).all(axis=1) & not_excluded
                flag = scores[sensors].gt(3).any(axis=1).astype(float).where(available)
                raw_flag = pd.Series(flag.to_numpy(), index=raw_trace["ROW_ID"])
                if DATA_KIND == "raw":
                    retro_flags[group] = raw_flag.reindex(retro_flags.index)
                else:
                    mapped = row_map[["MINUTE_ROW_ID"]].copy()
                    mapped["FLAG"] = row_map["SOURCE_ROW_ID"].map(raw_flag)
                    counts = mapped.groupby("MINUTE_ROW_ID")["FLAG"].agg(["max", "count", "size"])
                    minute_flag = pd.Series(np.nan, index=counts.index)
                    minute_flag.loc[counts["max"].eq(1)] = 1
                    minute_flag.loc[counts["max"].eq(0) & counts["count"].eq(counts["size"])] = 0
                    retro_flags[group] = minute_flag.reindex(retro_flags.index)
                    retro_flags.loc[base.loc[base["EXCLUDED"], "ROW_ID"], group] = np.nan
            sigma_reference = pd.DataFrame(reference_rows)
            retro_status = "LOT 전체 원본 기록에서 계산한 ±3σ, 공통 패턴 해제 전 사후 결과"
        else:
            raw_trace = pd.DataFrame()
            retro_status = "원본 기록과 1분 연결표가 맞지 않아 Z-score 사후 비교 보류"

    retrospective_rows = []
    for group in groups:
        for lot, rows in base.groupby("LOT_NO", sort=False):
            sigma = rows["ROW_ID"].map(retro_flags[group])
            label = rows[group + "__LABEL"]
            known = sigma.isin([0, 1]) & label.isin([0, 1])
            retrospective_rows.append({
                "LOT_NO": lot, "GROUP": group, "ROWS": len(rows), "KNOWN_BOTH": int(known.sum()),
                "ZSCORE1": int((known & sigma.eq(1)).sum()), "LABEL1": int((known & label.eq(1)).sum()),
                "ZSCORE1_LABEL0": int((known & sigma.eq(1) & label.eq(0)).sum()),
                "ZSCORE0_LABEL1": int((known & sigma.eq(0) & label.eq(1)).sum()),
            })
    retro_summary = pd.DataFrame(retrospective_rows)
    retro_summary.to_csv(os.path.join(OUT_DIR, "retrospective_zscore_summary.csv"), index=False, encoding=ENC)
    sigma_reference.to_csv(os.path.join(OUT_DIR, "retrospective_sigma_reference.csv"), index=False, encoding=ENC)
    saved_flags = base[["ROW_ID", "LOT_NO", "MEAS_DT"]].copy()
    for group in groups:
        saved_flags[group + "__ZSCORE_CANDIDATE"] = saved_flags["ROW_ID"].map(retro_flags[group]).astype("Int64")
    saved_flags.to_csv(os.path.join(OUT_DIR, "retrospective_zscore.csv"), index=False, encoding=ENC)
    print(retro_status)
    retro_names = {"LOT_NO": "LOT", "GROUP": "구역", "ROWS": "전체 기록", "KNOWN_BOTH": "둘 다 판정한 기록", "ZSCORE1": "Z-score 후보", "LABEL1": "현재 구역 1", "ZSCORE1_LABEL0": "Z-score 1, 현재 0", "ZSCORE0_LABEL1": "Z-score 0, 현재 1"}
    display(retro_summary.loc[retro_summary["GROUP"].eq(PLOT_GROUP)].rename(columns=retro_names))


### 6. 후보가 이어진 시간과 기록 확인

- 노란 배경은 아래 표에서 고른 모델의 후보 구간임
  - 기존 라벨은 그래프 아래의 `기존 구역 라벨` 줄에서 확인함
  - 모델 후보가 없으면 노란 배경을 칠하지 않음
- `069 Z4`와 `121 Z6`에서 모델이 표시한 구간을 확인함
  - 두 모델이 판정한 행이 있는 실험 중 후보가 많은 실험을 먼저 보여줌
  - 모델 결과를 본 뒤 고른 사후 검토 구간이며 탐지 조건 변경이 아님
- 같은 관측 구간에서 후보가 연속한 기록만 묶음
  - 처음부터 마지막 관측까지의 시간이며 고장이 지속된 시간은 아님
- 부품교체는 작업번호로 정비이력에 연결함
  - 부품교체만의 시각이 없으므로 연결된 작업 시간을 사용함
- 정비유형과 고장 항목의 코드는 데이터사전에 적힌 뜻으로 표시함
  - 비어 있는 고장 항목은 `기록 없음`으로 표시함
- 후보 시작부터 작업 시작까지의 시간은 두 시각의 차이임
  - 음수면 작업 시작이 먼저였고, 양수면 후보 시작이 먼저였음
  - 고장을 그만큼 미리 예측했다는 뜻은 아님
- 아래 문장은 실제 정비 지시가 아닌 점검 제안임


In [ ]:
review_choices = []
period_records = []
record_notes = []
if RUN_READY:
    events = pd.read_csv(files["event"], encoding=ENC)
    events = events.loc[events["PLANT_CD"].eq("CR1") & events["EVT_TYPE"].eq("강종변경")].copy()
    events["EVT_DT"] = pd.to_datetime(events["EVT_DT"])
    maintenance = pd.read_csv(files["maintenance"], encoding=ENC)
    maintenance = maintenance.loc[maintenance["EQP_TAG"].eq("P1-CR1-CAL01")].copy()
    for column in ["DETC_DT", "STRT_DT", "END_DT"]:
        maintenance[column] = pd.to_datetime(maintenance[column])
    parts = pd.read_csv(files["parts"], encoding=ENC)
    part_master = pd.read_csv(files["part_master"], encoding=ENC)
    parts = parts.merge(part_master[["PART_CD", "PART_NM"]], on="PART_CD", how="left")
    repair = pd.read_csv(files["repair_plan"], encoding=ENC)
    repair = repair.loc[repair["PLANT_CD"].eq("CR1")].copy()
    repair["STRT_DT"] = pd.to_datetime(repair["STRT_DT"])
    repair["END_EXCLUSIVE"] = pd.to_datetime(repair["END_DT"]).dt.normalize() + pd.Timedelta(days=1)
    review_targets = [("240069", "Z4"), ("240121", "Z6")]
    for lot, group in review_targets:
        candidates = coverage.loc[coverage["LOT_NO"].eq(lot) & coverage["GROUP"].eq(group)].copy()
        candidates["BOTH_HAVE_OUTPUT"] = candidates["IF_SCORED"].gt(0) & candidates["RFC_SCORED"].gt(0)
        candidate_totals = []
        for row in candidates.itertuples(index=False):
            if_rows = if_predictions.loc[if_predictions["CASE"].eq(row.CASE) & if_predictions["ROLE"].eq(row.ROLE) & if_predictions["LOT_NO"].eq(lot)]
            rfc_rows = rfc_predictions.loc[rfc_predictions["CASE"].eq(row.CASE) & rfc_predictions["ROLE"].eq(row.ROLE) & rfc_predictions["LOT_NO"].eq(lot)]
            candidate_totals.append(int(if_rows[group + "__FLAG"].eq(1).sum() + rfc_rows[group + "__FLAG"].eq(1).sum()))
        candidates["CANDIDATE_TOTAL"] = candidate_totals
        candidates = candidates.sort_values(["BOTH_HAVE_OUTPUT", "CANDIDATE_TOTAL", "CASE"], ascending=[False, False, True], kind="stable")
        if candidates.empty:
            continue
        choice = candidates.iloc[0]
        case, role = choice["CASE"], choice["ROLE"]
        source = base.loc[base["LOT_NO"].eq(lot)].copy()
        if_rows = if_predictions.loc[if_predictions["CASE"].eq(case) & if_predictions["ROLE"].eq(role) & if_predictions["LOT_NO"].eq(lot)].copy()
        rfc_rows = rfc_predictions.loc[rfc_predictions["CASE"].eq(case) & rfc_predictions["ROLE"].eq(role) & rfc_predictions["LOT_NO"].eq(lot)].copy()
        for model_name, prediction_rows in [("IsolationForest", if_rows), ("RandomForestClassifier", rfc_rows)]:
            source[model_name] = source["ROW_ID"].map(prediction_rows.set_index("ROW_ID")[group + "__FLAG"])
        source["RETROSPECTIVE_Z"] = source["ROW_ID"].map(retro_flags[group])
        for suffix in ["DUMMY0", "DUMMY_RANDOM0"]:
            source[suffix] = source["ROW_ID"].map(rfc_rows.set_index("ROW_ID")[group + "__" + suffix])
        source["REVIEW_SEGMENT"] = source["ROW_ID"].map(if_rows.set_index("ROW_ID")["SEGMENT"])
        source["CONTEXT"] = source["ROW_ID"].map(if_rows.set_index("ROW_ID")["CONTEXT"])
        episodes = []
        for model_name in ["IsolationForest", "RandomForestClassifier"]:
            flag = source[model_name].eq(1)
            change = flag.ne(flag.shift()) | source["REVIEW_SEGMENT"].ne(source["REVIEW_SEGMENT"].shift())
            for _, block in source.groupby(change.cumsum(), sort=False):
                if not flag.loc[block.index[0]]:
                    continue
                episode = {
                    "CASE": case, "ROLE": role, "LOT_NO": lot, "GROUP": group, "MODEL": model_name,
                    "START": block["OBS_START"].iloc[0], "END": block["OBS_END"].iloc[-1],
                    "CANDIDATE_ROWS": len(block),
                    "OBSERVED_SPAN_SECONDS": (block["OBS_END"].iloc[-1] - block["OBS_START"].iloc[0]).total_seconds(),
                    "LABEL1_ROWS": int(block[group + "__LABEL"].eq(1).sum()),
                    "CONTEXT": ", ".join(block["CONTEXT"].dropna().astype(str).unique()),
                }
                episodes.append(episode)
                period_records.append(episode)
        episode_table = pd.DataFrame(episodes)
        if len(episode_table):
            episode_table = episode_table.sort_values(["CANDIDATE_ROWS", "START"], ascending=[False, True], kind="stable")
            first = episode_table.iloc[0]
            start, end = first["START"], first["END"]
            selected_model = first["MODEL"]
        else:
            start, end = source["OBS_START"].min(), source["OBS_END"].max()
            selected_model = "후보 없음"
        view_start = max(source["OBS_START"].min(), start - pd.Timedelta(minutes=VIEW_MARGIN_MINUTES))
        view_end = min(source["OBS_END"].max(), end + pd.Timedelta(minutes=VIEW_MARGIN_MINUTES))
        review_choices.append({"LOT_NO": lot, "GROUP": group, "CASE": case, "ROLE": role, "VIEW_START": view_start, "VIEW_END": view_end, "SELECTED_START": start, "SELECTED_END": end, "SELECTED_MODEL": selected_model, "SELECTION": "결과를 본 뒤 두 모델 판정이 있는 실험 우선, 후보 기록 수가 많은 실험과 구간부터 사후 확인"})

        # 그림과 겹친 작업, 감지 기록, 앞뒤 가장 가까운 작업을 구분함
        overlap = maintenance["STRT_DT"].le(view_end) & maintenance["END_DT"].ge(view_start)
        detected = maintenance["DETC_DT"].between(view_start, view_end)
        near = maintenance.loc[overlap | detected].copy()
        near["RELATION"] = "그림에서 감지되었거나 작업 시간이 겹침"
        before = maintenance.loc[maintenance["END_DT"].lt(view_start)].sort_values("END_DT").tail(1).copy()
        before["RELATION"] = "그림보다 앞선 가장 가까운 작업"
        after = maintenance.loc[maintenance["STRT_DT"].gt(view_end)].sort_values("STRT_DT").head(1).copy()
        after["RELATION"] = "그림보다 뒤의 가장 가까운 작업"
        near = pd.concat([near, before, after], ignore_index=True).drop_duplicates("WO_NO").sort_values("STRT_DT")
        near["FROM_SELECTED_START_HOURS"] = (near["STRT_DT"] - start).dt.total_seconds() / 3600
        part_text = []
        for work_id in near["WO_NO"]:
            descriptions = []
            for part in parts.loc[parts["WO_NO"].eq(work_id)].itertuples(index=False):
                descriptions.append(str(part.PART_NM) + " " + str(part.QTY) + "개, " + str(part.CHG_RSN))
            part_text.append(" / ".join(descriptions) if descriptions else "연결된 부품교체 기록 없음")
        near["PARTS"] = part_text
        near["LOT_REVIEW"] = lot
        near["GROUP_REVIEW"] = group
        record_notes.extend(near.to_dict("records"))
        displayed_events = events.loc[events["EVT_DT"].between(view_start, view_end)]
        preceding_event = events.loc[events["EVT_DT"].lt(view_start)].sort_values("EVT_DT").tail(1)
        displayed_events = pd.concat([preceding_event, displayed_events]).drop_duplicates("EVT_ID")
        displayed_plans = repair.loc[repair["STRT_DT"].le(view_end) & repair["END_EXCLUSIVE"].gt(view_start)]

        display(Markdown("### LOT " + lot + " / " + group + " / " + method_names.get(str(choice["METHOD"]), str(choice["METHOD"]))))
        result_table = metrics.loc[metrics["CASE"].eq(case) & metrics["ROLE"].eq(role) & metrics["LOT_NO"].eq(lot) & metrics["GROUP"].eq(group), metric_columns]
        display(result_table.replace({"METHOD": method_names}).rename(columns=show_names).round(4))
        display(candidates.loc[candidates["CASE"].eq(case), ["ALL_ROWS", "IF_SCORED", "RFC_SCORED", "COMMON_ROWS", "LABEL1_NOT_COMPARED", "REASON"]].rename(columns=show_names))
        if len(episode_table):
            display(episode_table.head(8).rename(columns={"MODEL": "모델", "START": "후보 시작", "END": "후보 끝", "CANDIDATE_ROWS": "후보 기록 수", "OBSERVED_SPAN_SECONDS": "처음부터 마지막까지 초", "LABEL1_ROWS": "기존 구역 1", "CONTEXT": "작업 기록"}).drop(columns=["CASE", "ROLE", "LOT_NO", "GROUP"]))
        else:
            print("이 실험에서 후보 구간이 없었음. 보류와 후보 0은 결과표에서 구분할 것.")

        view = source.loc[source["OBS_END"].ge(view_start) & source["OBS_START"].le(view_end)].copy()
        known_retro = view["RETROSPECTIVE_Z"].isin([0, 1]) & view[group + "__LABEL"].isin([0, 1])
        display(pd.DataFrame([{
            "아래 그림의 첫 시각": view_start, "아래 그림의 마지막 시각": view_end,
            "Z-score와 라벨이 모두 있는 기록": int(known_retro.sum()),
            "LOT 전체 Z-score 후보": int((known_retro & view["RETROSPECTIVE_Z"].eq(1)).sum()),
            "현재 구역 라벨 1": int((known_retro & view[group + "__LABEL"].eq(1)).sum()),
            "Z-score 1, 현재 0": int((known_retro & view["RETROSPECTIVE_Z"].eq(1) & view[group + "__LABEL"].eq(0)).sum()),
        }]))
        sensors = groups[group]
        fig, axes = plt.subplots(len(sensors) + 1, 1, figsize=(14, len(sensors) * 2.1 + 5.0), sharex=True, gridspec_kw={"height_ratios": [2] * len(sensors) + [3.2]})
        time_gap = view["MEAS_DT"].diff().dt.total_seconds()
        draw_break = view["REVIEW_SEGMENT"].ne(view["REVIEW_SEGMENT"].shift()) | time_gap.le(0)
        draw_break = draw_break | time_gap.gt(60 if DATA_KIND == "minute" else 15)
        view["DRAW_SEGMENT"] = draw_break.cumsum()
        for axis, sensor in zip(axes[:-1], sensors):
            for _, segment_rows in view.groupby("DRAW_SEGMENT", sort=False):
                axis.plot(segment_rows["MEAS_DT"], segment_rows[sensor], color="#426b84", linewidth=0.8)
            unit = "℃" if sensor.startswith("TC-") else "%" if sensor.startswith("OP-") else "기록값"
            axis.set_ylabel(sensor + "\n(" + unit + ")")
            axis.grid(alpha=0.15)
            if selected_model != "후보 없음":
                axis.axvspan(start, end, color="#f4c24a", alpha=0.13)
            for event in displayed_events.itertuples(index=False):
                if view_start <= event.EVT_DT <= view_end:
                    axis.axvline(event.EVT_DT, color="#9370ad", linestyle="--", linewidth=0.9)
            for work in near.itertuples(index=False):
                if view_start <= work.DETC_DT <= view_end:
                    axis.axvline(work.DETC_DT, color="#bf623f", linestyle=":", linewidth=0.9)
                if view_start <= work.STRT_DT <= view_end:
                    axis.axvline(work.STRT_DT, color="#368ca6", linestyle="--", linewidth=0.9)
                if view_start <= work.END_DT <= view_end:
                    axis.axvline(work.END_DT, color="#368ca6", linestyle=":", linewidth=0.9)
        # 원본 기록은 점으로 표시하고, 1분 기록은 실제 관측 범위만 칠함
        strips = [
            (group + "__LABEL", "기존 구역 라벨"),
            ("RETROSPECTIVE_Z", "LOT 전체 Z-score (사후)"),
            ("IsolationForest", "IsolationForest"),
            ("RandomForestClassifier", "RandomForestClassifier"),
            ("DUMMY0", "Dummy 0 고정"),
            ("DUMMY_RANDOM0", "Dummy 무작위 시드 0"),
        ]
        for strip_number, (column, title) in enumerate(strips):
            if DATA_KIND == "minute":
                for _, observation in view.iterrows():
                    value = observation[column]
                    color = "#d7dce1" if pd.isna(value) else "#d6a100" if value == 1 else "#fff9df"
                    axes[-1].fill_between([observation["OBS_START"], observation["OBS_END"]], strip_number - 0.4, strip_number + 0.4, color=color, linewidth=0)
            else:
                for state, color in [(0, "#fff9df"), (1, "#d6a100"), (None, "#d7dce1")]:
                    selected = view[column].isna() if state is None else view[column].eq(state)
                    axes[-1].plot(view.loc[selected, "MEAS_DT"], np.full(int(selected.sum()), strip_number), linestyle="none", marker="|", color=color, markersize=14)
        axes[-1].set_yticks(range(len(strips)), [title for column, title in strips])
        axes[-1].set_ylim(len(strips) - 0.4, -0.6)
        axes[-1].set_xlim(view_start, view_end)
        axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
        axes[-1].set_xlabel(str(view_start.year) + "년 실제 기록 시각" + (" / 분의 값은 AVAILABLE_AT 이후 사용 가능" if DATA_KIND == "minute" else ""))
        plot_title = "후보 구간 없음" if selected_model == "후보 없음" else "노란 배경: " + selected_model + " 후보 구간"
        plot_method = method_names.get(str(choice["METHOD"]), str(choice["METHOD"]))
        fig.suptitle("LOT " + lot + " | " + group + " | " + plot_method + "\n" + plot_title)
        fig.subplots_adjust(left=0.20, right=0.98, top=0.92, bottom=0.17, hspace=0.15)
        fig.text(0.20, 0.07, "아래 띠: 노랑=후보 1, 아주 연한 노랑=0, 회색=보류. 흰 시간 공백은 관측 없음.", fontsize=9)
        fig.text(0.20, 0.025, "세로선: 보라 파선=강종변경, 갈색 점선=정비 감지, 파랑 파선=작업 시작, 파랑 점선=작업 종료", fontsize=9)
        fig.savefig(os.path.join(FIG_DIR, "review_" + lot + "_" + group + ".png"), dpi=130, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        print("그림 앞선 강종변경과 그림 안 변경 기록")
        display(displayed_events[["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL"]].rename(columns={"EVT_ID": "조업 기록 번호", "EVT_DT": "변경 시각", "BEF_VAL": "이전 강종", "AFT_VAL": "변경 후 강종"}))
        print("같은 소둔로 작업 기록. 시간상 가까운 기록이며 후보의 원인으로 확정하지 않음.")
        print(context_generation)
        # 화면에 보여줄 복사본만 풀어 씀. 저장하는 정비 원본 코드는 유지함.
        work_view = near[["WO_NO", "RELATION", "DETC_DT", "STRT_DT", "END_DT", "MNT_TYPE", "FAIL_MODE", "REMARK", "PARTS", "FROM_SELECTED_START_HOURS"]].copy()
        work_view["MNT_TYPE"] = work_view["MNT_TYPE"].replace({
            "BM": "사후정비 (BM)",
            "CBM": "상태기반정비 (CBM)",
            "TBM": "시간기준정비 (TBM)",
        })
        work_view["FAIL_MODE"] = work_view["FAIL_MODE"].replace({
            "BRG": "베어링 (BRG)",
            "UNB": "불평형 (UNB)",
            "MAL": "정렬불량 (MAL)",
            "LUB": "윤활 (LUB)",
            "ELE": "전기 (ELE)",
            "SEL": "실링 (SEL)",
            "WER": "마모 (WER)",
            "ETC": "기타 (ETC)",
        }).fillna("기록 없음")
        display(work_view.rename(columns={
            "WO_NO": "작업지시번호",
            "RELATION": "그림과의 관계",
            "DETC_DT": "감지 시각",
            "STRT_DT": "작업 시작",
            "END_DT": "작업 끝",
            "MNT_TYPE": "정비유형",
            "FAIL_MODE": "정비기록의 고장 항목",
            "REMARK": "작업 메모",
            "PARTS": "연결된 부품교체",
            "FROM_SELECTED_START_HOURS": "후보 시작부터 작업 시작까지 시간",
        }))
        if len(displayed_plans):
            print("그림 시간과 겹친 정기수리 계획")
            display(displayed_plans)
        if group == "Z4":
            print("점검 제안: CP-Z4의 상승 폭과 돌아오는 시간을 TC-Z4, OP-Z4 변화와 비교할 것. CP 하나만 보고 히터 고장으로 단정하지 않음.")
        else:
            if not raw_trace.empty:
                traced = raw_trace.loc[raw_trace["LOT_NO"].eq(lot) & raw_trace["MEAS_DT"].between(start, end)].copy()
                reviewed = traced.loc[traced["Z6__COMMON_REVIEW_IDS"].fillna("").ne("")].copy()
                trace_title = "LOT 전체 시간" if selected_model == "후보 없음" else "선택한 후보 시간"
                print(trace_title + "과 공통 패턴 검토가 겹친 원본 기록:", len(reviewed), "개")
                trace_table = reviewed.groupby("Z6__COMMON_REVIEW_IDS", sort=False).agg(
                    FIRST=("MEAS_DT", "min"), LAST=("MEAS_DT", "max"), ROWS=("ROW_ID", "size"),
                    TC_MIN=("TC-Z6", "min"), TC_MAX=("TC-Z6", "max"), OP_MIN=("OP-Z6", "min"), OP_MAX=("OP-Z6", "max"),
                ).reset_index()
                trace_table.to_csv(os.path.join(OUT_DIR, "common_pattern_overlap.csv"), index=False, encoding=ENC)
                display(trace_table.rename(columns={"Z6__COMMON_REVIEW_IDS": "공통 패턴 검토 기록", "FIRST": "겹친 첫 관측", "LAST": "겹친 마지막 관측", "ROWS": "원본 기록 수", "TC_MIN": "TC-Z6 최저", "TC_MAX": "TC-Z6 최고", "OP_MIN": "OP-Z6 최저", "OP_MAX": "OP-Z6 최고"}))
                print("겹친 원본 기록의 구역 라벨: 0 =", int(reviewed["Z6__LABEL"].eq(0).sum()), "/ 1 =", int(reviewed["Z6__LABEL"].eq(1).sum()))
            else:
                print("v4 생성에 사용한 원본 기록 파일을 찾지 못해 공통 패턴의 초 단위 겹침은 보류함.")
            print("점검 제안: TC-Z6가 내려갈 때 OP-Z6가 올라가는 제어 반응과 복귀 시간을 같은 강종의 앞선 LOT와 비교할 것. 반복되는 모양 자체를 고장으로 단정하지 않음.")
            print("공통 패턴 해제는 검토한 센서와 시간에만 적용했음. 모양이 비슷한 다른 구간까지 정상으로 확정하지 않음.")
        print("정비와 부품교체의 시간 관계는 확인하되 직접 원인 연결은 별도로 확인해야 함.")

    pd.DataFrame(period_records).to_csv(os.path.join(OUT_DIR, "candidate_periods.csv"), index=False, encoding=ENC)
    pd.DataFrame(review_choices).to_csv(os.path.join(OUT_DIR, "reviewed_windows.csv"), index=False, encoding=ENC)
    pd.DataFrame(record_notes).to_csv(os.path.join(OUT_DIR, "maintenance_parts_context.csv"), index=False, encoding=ENC)


### 7. 원본 기록과 1분 결과를 읽는 방법

- v3 한 행은 원본 기록이며 v4 한 행은 같은 구간 안에서 모은 한 분의 기록임
  - 행수나 Precision과 Recall을 같은 분모처럼 직접 비교하지 않음
- 각 CSV의 같은 학습 방법과 구역 안에서 같은 평가 행으로 모델을 비교했음
  - v4를 원본 기록에 연결할 때는 `review/v4/raw_to_minute.csv`를 사용함
  - 원본 기록과 1분 기록을 같은 `ROW_ID`로 붙이지 않음
- 원본 기록과 1분 모델은 학습 기준 기록 수가 다름
  - v3는 유효한 원본 기록 `256개` 이상이며 서로 다른 관측 `5분` 이상이어야 함
  - v4는 유효한 `1분 기록 5개` 이상이며 서로 다른 관측 `5분` 이상이어야 함
  - RandomForestClassifier의 `min_samples_leaf=5`는 나무의 마지막 판단 칸에 최소 기록 `5개`를 남기는 설정임
  - 같은 `5개`여도 v3와 v4에서 담는 시간이 다름
  - 결과 차이를 평균으로 묶은 효과 하나로만 설명하지 않음
- `058` 정기수리 전 정지와 `061` 재가동은 팀 확정 상태를 유지함
  - 모델 후보와 팀이 정한 운전 상태는 서로 다른 정보임
- 점검 제안은 실제 정비 결정이 아님
  - 모델 후보와 정비 기록이 가까운 것만으로 원인을 확정하지 않음


In [ ]:
if RUN_READY:
    display(pd.DataFrame([{
        "CSV": DATA_FILE, "행 단위": "1분 안의 같은 관측 구간" if DATA_KIND == "minute" else "원본 기록",
        "입력 행수": len(base), "비교 기준": "같은 학습 방법, 구역, 역할, 관측 행",
        "원본 기록 연결표": "review/v4/raw_to_minute.csv" if DATA_KIND == "minute" else "ROW_ID",
    }]))
    selected_config = {}
    for model_name, run in selected_runs.items():
        selected_config[model_name] = {"folder": run["folder"], "run_id": run["run_id"], "config_sha256": run["config_sha256"]}
    review_settings = {
        "issue": 101, "input_file": DATA_FILE, "input_sha256": input_hashes,
        "data_kind": DATA_KIND, "run_id": RUN_ID, "completed_at": datetime.now().isoformat(),
        "selected_runs": selected_config, "shared_cases": common_cases, "groups": groups,
        "context_generation": context_generation,
        "comparison_keys": ["CASE", "GROUP", "ROLE", "ROW_ID"],
        "comparison_policy": "known matching group label and all four saved predictions, same source and context",
        "common_row_hashes": common_row_hashes,
        "dummy_random_main_table_seed": 0,
        "dummy_repeats": "saved RFC metrics only; original RFC denominator, not recomputed on IF intersection",
        "label_interpretation": "retrospective complete LOT candidate-rule agreement, not independent fault truth",
        "zscore_role": "separate complete LOT raw mean/std(ddof0), strict abs(z)>3 before common-pattern removals; not prospective baseline or metric ranking",
        "zscore_status": retro_status,
        "minute_zscore": "aggregate mapped raw candidate flags; never recompute sigma from minute means",
        "new_model_fit": False, "threshold_changed": False,
        "view_margin_minutes": VIEW_MARGIN_MINUTES,
        "episode_duration": "elapsed span between first and last candidate observations; not failure duration",
        "inspection_proposal_only": True, "minute_raw_rows_directly_merged": False,
    }
    with open(os.path.join(OUT_DIR, "review_settings.json"), "w", encoding="utf-8") as settings_file:
        json.dump(review_settings, settings_file, ensure_ascii=False, indent=2)
    print("비교표와 확인 구간:", OUT_DIR)
    print("그림:", FIG_DIR)
